# Docker for Machine Learning
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/11_MLOps_Deployment/docker_for_ml.ipynb)

'Works on my machine' ends here. A Docker image packages your model + code + exact Python version + system libs into one portable unit that runs identically on any host - the foundation of reproducible ML deployment.

This notebook writes all project files, then builds/runs when Docker is available.

## 1. Project layout we will create

In [ ]:
project_files = """
ml-service/
|-- app.py               # FastAPI serving the model
|-- train.py             # trains & saves model.pkl
|-- requirements.txt     # pinned deps
|-- Dockerfile           # the recipe
`.dockerignore           # keep images small
"""
print(project_files)

## 2. Write every file from the notebook

In [ ]:
import os, pickle, textwrap
os.makedirs("ml-service", exist_ok=True)

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
X, y = load_iris(return_X_y=True)
pickle.dump(LogisticRegression(max_iter=1000).fit(X, y),
            open("ml-service/model.pkl", "wb"))

open("ml-service/requirements.txt", "w").write(
    "fastapi==0.115.*\nuvicorn==0.34.*\nscikit-learn==1.6.*\n")

open("ml-service/.dockerignore", "w").write("__pycache__\n*.pyc\n.git\n")

open("ml-service/app.py", "w").write(textwrap.dedent("""
    from fastapi import FastAPI
    from pydantic import BaseModel
    import pickle

    app = FastAPI(title="iris-docker")
    model = pickle.load(open("model.pkl", "rb"))

    class Feats(BaseModel):
        sl: float; sw: float; pl: float; pw: float

    @app.post("/predict")
    def predict(f: Feats):
        p = int(model.predict([[f.sl, f.sw, f.pl, f.pw]])[0])
        return {"species": ["setosa", "versicolor", "virginica"][p]}
"""))
print("files written:", os.listdir("ml-service"))

## 3. The Dockerfile, line by line

In [ ]:
dockerfile = """
FROM python:3.11-slim            # tiny base with python preinstalled
WORKDIR /app                     # cwd inside container
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # deps layer first = cached rebuilds
COPY . .
EXPOSE 8000
HEALTHCHECK CMD curl -f http://localhost:8000/ || exit 1
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
open("ml-service/Dockerfile", "w").write(dockerfile.replace('            ', ''))
print(dockerfile)

## 4. Build, run, test (needs Docker installed)

In [ ]:
commands = """
cd ml-service

docker build -t iris-api:v1 .

docker run -d --name iris -p 8000:8000 iris-api:v1

curl http://localhost:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"sl":5.1,"sw":3.5,"pl":1.4,"pw":0.2}'

docker logs iris --tail 5
docker stop iris && docker rm iris
"""
print(commands)

## Production habits
| Habit | Why |
|---|---|
| pin versions (`==`) | rebuilds stay identical |
| deps layer BEFORE code layer | code edits rebuild in seconds |
| `.dockerignore` | smaller context, fewer leaks |
| non-root `USER appuser` | container escape hardening |
| tag images (`v1`, git-sha) | instant rollback |
| scan (`docker scout cves`) | catch vulnerable bases |

GPU inference? Swap base for `nvidia/cuda:*-runtime` + `--gpus all`. Compose multi-service stacks (api + redis + monitoring) with `docker-compose.yml`.